# TrueShift V-JEPA Inference Node

This notebook runs the V-JEPA model and exposes it via Ngrok for the TrueShift Backend.

In [ ]:
!pip install flask pyngrok opencv-python-headless torch torchvision

In [ ]:
import os
import requests
from flask import Flask, request, jsonify
from pyngrok import ngrok
import random

# CONFIG
BACKEND_URL = "https://api.trueshift.app" # CHANGE THIS to local if tunneling: http://xxxx.ngrok.io
BACKEND_SECRET = "change-me-in-production"
NGROK_AUTH_TOKEN = "" # ADD YOUR TOKEN

app = Flask(__name__)

# MODEL LOADER (Placeholder)
# In real implementation, load V-JEPA model here
print("Loading V-JEPA Model...")
model = None 

@app.route('/predict_chunk', methods=['POST'])
def predict_chunk():
    if 'file' not in request.files:
        return jsonify({"error": "No file part"}), 400
    
    file = request.files['file']
    # Save for processing
    filename = "temp_chunk.mp4"
    file.save(filename)
    
    # MOCK INFERENCE LOGIC
    # Replace with actual V-JEPA + Classifier
    score = random.randint(70, 100)
    feedback = "Good form" if score > 85 else "Check your posture"
    
    return jsonify({
        "status": "success",
        "data": {
            "form_score": score,
            "feedback": feedback,
            "issues": []
        }
    })

def start_server():
    # Open ngrok tunnel
    if NGROK_AUTH_TOKEN:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    public_url = ngrok.connect(5000).public_url
    print(" * Ngrok Tunnel:", public_url)
    
    # Auto-Register with Backend
    try:
        reg_url = f"{BACKEND_URL}/api/v1/vision/internal/register"
        requests.post(reg_url, json={"url": public_url, "secret": BACKEND_SECRET})
        print(" * Registered with Backend successfully.")
    except Exception as e:
        print(" * Registration failed:", e)

    app.run(port=5000)

if __name__ == '__main__':
    start_server()